<a href="https://colab.research.google.com/github/sarawutking/vibranode-bridge/blob/main/Project_CE312.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipywidgets pandas numpy scikit-learn requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import requests, zipfile, io, os
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("📥 กำลังโหลด MovieLens Dataset...")

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall(".")

movies_df  = pd.read_csv("ml-latest-small/movies.csv")
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")

print(f"✅ โหลดสำเร็จ! หนังทั้งหมด {len(movies_df)} เรื่อง | Rating ทั้งหมด {len(ratings_df)} รายการ")

avg_ratings = ratings_df.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

movies_df = movies_df.merge(avg_ratings, on="movieId", how="left")
movies_df["avg_rating"]   = movies_df["avg_rating"].fillna(0).round(2)
movies_df["rating_count"] = movies_df["rating_count"].fillna(0).astype(int)

movies_df["genres_list"] = movies_df["genres"].apply(lambda x: x.split("|"))

print("✅ เตรียมข้อมูลเสร็จสิ้น!")

📥 กำลังโหลด MovieLens Dataset...
✅ โหลดสำเร็จ! หนังทั้งหมด 9742 เรื่อง | Rating ทั้งหมด 100836 รายการ
✅ เตรียมข้อมูลเสร็จสิ้น!


In [ ]:
def merge_sort(arr, key="avg_rating", reverse=True):
    """
    Merge Sort — เรียงลำดับ list of dict ตาม key ที่กำหนด
    Time Complexity: O(n log n)
    """
    if len(arr) <= 1:
        return arr

    mid   = len(arr) // 2
    left  = merge_sort(arr[:mid],  key=key, reverse=reverse)
    right = merge_sort(arr[mid:],  key=key, reverse=reverse)

    return merge(left, right, key, reverse)


def merge(left, right, key, reverse):
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        left_val  = left[i][key]
        right_val = right[j][key]

        if (left_val >= right_val) if reverse else (left_val <= right_val):
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

In [ ]:
def binary_search_movie(sorted_movies, query):

    query = query.lower().strip()
    low, high = 0, len(sorted_movies) - 1
    found_idx = -1

    while low <= high:
        mid = (low + high) // 2
        title = sorted_movies[mid]["title"].lower()
        if title >= query:
            found_idx = mid
            high = mid - 1
        else:
            low = mid + 1

    if found_idx == -1:
        return []

    results = []
    for i in range(max(0, found_idx - 5), min(len(sorted_movies), found_idx + 200)):
        if query in sorted_movies[i]["title"].lower():
            results.append(sorted_movies[i])

    return results

In [ ]:
ALL_GENRES = ["Action","Adventure","Animation","Children","Comedy","Crime",
    "Documentary","Drama","Fantasy","Film-Noir","Horror","IMAX",
    "Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"]

def build_feature_matrix(df):
    """สร้าง feature vector ต่อหนัง 1 เรื่อง = [genre flags...] + [normalized avg_rating]"""
    genre_matrix = np.zeros((len(df), len(ALL_GENRES)), dtype=float)
    for i, genres in enumerate(df["genres_list"]):
        for g in genres:
            if g in ALL_GENRES:
                genre_matrix[i, ALL_GENRES.index(g)] = 1.0

    ratings_norm = MinMaxScaler().fit_transform(df[["avg_rating"]].values)
    return np.hstack([genre_matrix, ratings_norm])


def train_knn(df, n_neighbors=11):
    """Train KNN model"""
    features = build_feature_matrix(df)
    model = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
    model.fit(features)
    return model, features


def get_recommendations(movie_title, df, model, features, top_n=5):

    matches = df[df["title"].str.lower().str.contains(movie_title.lower())]
    if matches.empty:
        return None, []

    movie_row = matches.iloc[0]
    idx = df.index.get_loc(movie_row.name)

    distances, indices = model.kneighbors([features[idx]])
    rec_indices = [i for i in indices[0] if i != idx][:top_n]

    recommendations = df.iloc[rec_indices][["title","genres","avg_rating","rating_count"]].copy()
    recommendations["similarity"] = [round((1 - distances[0][j+1]) * 100, 1)
                                      for j in range(len(rec_indices))]
    return movie_row, recommendations.to_dict("records")

In [ ]:
print("🔧 กำลัง Train KNN Model...")
movies_clean = movies_df.dropna(subset=["avg_rating"]).reset_index(drop=True)
knn_model, feature_matrix = train_knn(movies_clean)

movies_list = movies_clean.to_dict("records")
sorted_by_rating = merge_sort(movies_list, key="avg_rating", reverse=True)

sorted_by_title = sorted(movies_list, key=lambda x: x["title"].lower())

print("✅ ระบบพร้อมใช้งานแล้ว!")


🔧 กำลัง Train KNN Model...
✅ ระบบพร้อมใช้งานแล้ว!


In [ ]:
def create_ui():

    # ── CSS Styling ──────────────────────────────────────────
    display(HTML("""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Bebas+Neue&family=Inter:wght@300;400;600&display=swap');

      .movie-app { font-family: 'Inter', sans-serif; }

      .app-title {
        font-family: 'Bebas Neue', sans-serif;
        font-size: 3em;
        background: linear-gradient(135deg, #f5c518, #e50914);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin: 0; letter-spacing: 2px;
      }

      .section-header {
        font-family: 'Bebas Neue', sans-serif;
        font-size: 1.4em;
        color: #f5c518;
        border-left: 4px solid #e50914;
        padding-left: 10px;
        margin: 16px 0 8px;
        letter-spacing: 1px;
      }

      .movie-card {
        background: #1a1a2e;
        border: 1px solid #2d2d44;
        border-radius: 10px;
        padding: 12px 16px;
        margin: 6px 0;
        display: flex;
        justify-content: space-between;
        align-items: center;
        transition: border-color 0.2s;
      }

      .movie-card:hover { border-color: #f5c518; }

      .movie-title { color: #ffffff; font-weight: 600; font-size: 0.95em; }
      .movie-genres { color: #888; font-size: 0.78em; margin-top: 3px; }

      .badge-rating {
        background: #f5c518;
        color: #1a1a2e;
        border-radius: 20px;
        padding: 3px 10px;
        font-weight: 700;
        font-size: 0.85em;
        white-space: nowrap;
      }

      .badge-sim {
        background: #e50914;
        color: white;
        border-radius: 20px;
        padding: 3px 10px;
        font-weight: 700;
        font-size: 0.85em;
        white-space: nowrap;
      }

      .rank-num {
        color: #f5c518;
        font-family: 'Bebas Neue', sans-serif;
        font-size: 1.3em;
        min-width: 32px;
      }

      .card-left { display: flex; align-items: center; gap: 12px; }
      .info-box {
        background: #0f3460;
        border-radius: 8px;
        padding: 10px 16px;
        margin: 8px 0;
        color: #ccc;
        font-size: 0.85em;
        border-left: 3px solid #f5c518;
      }
      body, .widget-area { background: #0d0d1a !important; }
    </style>

    <div class='movie-app'>
      <p class='app-title'>🎬 CineMatch</p>
      <p style='color:#888; margin:0 0 20px; font-size:0.9em'>
        Movie Recommendation System — CE332 Project
      </p>
    </div>
    """))

    # ── Tab Layout ────────────────────────────────────────────
    tab = widgets.Tab()

    # ═══════════════════════════════════════════════════════
    #  TAB 1: Top Rated (Merge Sort)
    # ═══════════════════════════════════════════════════════
    sort_out    = widgets.Output()
    sort_slider = widgets.IntSlider(value=10, min=5, max=50, step=5,
                                    description="แสดง:", style={"description_width":"55px"})
    sort_btn    = widgets.Button(description="🔃 เรียงลำดับ (Merge Sort)",
                                 button_style="warning",
                                 layout=widgets.Layout(width="240px"))

    def on_sort_click(_):
        with sort_out:
            clear_output()
            n = sort_slider.value
            top = sorted_by_rating[:n]
            html = f"<div class='section-header'>🏆 Top {n} หนังยอดนิยม (เรียงด้วย Merge Sort)</div>"
            html += "<div class='info-box'>⚙️ <b>Merge Sort</b>: แบ่ง list ครึ่งซ้าย-ขวาซ้ำๆ แล้ว merge กลับ ตาม avg_rating จากมากไปน้อย &nbsp;|&nbsp; Time: O(n log n)</div>"
            for i, m in enumerate(top, 1):
                count = m.get("rating_count", 0)
                html += f"""
                <div class='movie-card'>
                  <div class='card-left'>
                    <span class='rank-num'>#{i}</span>
                    <div>
                      <div class='movie-title'>{m['title']}</div>
                      <div class='movie-genres'>{m['genres']} &nbsp;·&nbsp; {count:,} ratings</div>
                    </div>
                  </div>
                  <span class='badge-rating'>⭐ {m['avg_rating']}</span>
                </div>"""
            display(HTML(html))

    sort_btn.on_click(on_sort_click)
    tab1 = widgets.VBox([
        widgets.HBox([sort_slider, sort_btn]),
        sort_out
    ])

    # ═══════════════════════════════════════════════════════
    #  TAB 2: ค้นหา (Binary Search)
    # ═══════════════════════════════════════════════════════
    search_out   = widgets.Output()
    search_input = widgets.Text(placeholder="พิมพ์ชื่อหนัง เช่น Toy Story, Batman...",
                                description="ค้นหา:",
                                style={"description_width":"55px"},
                                layout=widgets.Layout(width="380px"))
    search_btn   = widgets.Button(description="🔍 ค้นหา (Binary Search)",
                                  button_style="info",
                                  layout=widgets.Layout(width="220px"))

    def on_search_click(_):
        with search_out:
            clear_output()
            query = search_input.value.strip()
            if not query:
                display(HTML("<p style='color:#e50914'>⚠️ กรุณาพิมพ์ชื่อหนัง</p>"))
                return

            results = binary_search_movie(sorted_by_title, query)
            html = f"<div class='section-header'>🔍 ผลการค้นหา: \"{query}\" ({len(results)} เรื่อง)</div>"
            html += "<div class='info-box'>⚙️ <b>Binary Search</b>: แบ่ง list ที่ sort ตามชื่อแล้ว ค้นหาจุดเริ่มต้น จากนั้น scan รวบรวม partial match &nbsp;|&nbsp; Time: O(log n)</div>"

            if not results:
                html += "<p style='color:#888'>ไม่พบหนังที่ตรงกัน ลองพิมพ์ชื่ออื่น</p>"
            else:
                for m in results[:15]:
                    count = m.get("rating_count", 0)
                    html += f"""
                    <div class='movie-card'>
                      <div class='card-left'>
                        <div>
                          <div class='movie-title'>{m['title']}</div>
                          <div class='movie-genres'>{m['genres']} &nbsp;·&nbsp; {count:,} ratings</div>
                        </div>
                      </div>
                      <span class='badge-rating'>⭐ {m['avg_rating']}</span>
                    </div>"""
            display(HTML(html))

    search_btn.on_click(on_search_click)
    tab2 = widgets.VBox([
        widgets.HBox([search_input, search_btn]),
        search_out
    ])

    # ═══════════════════════════════════════════════════════
    #  TAB 3: แนะนำหนัง (KNN)
    # ═══════════════════════════════════════════════════════
    rec_out   = widgets.Output()
    rec_input = widgets.Text(placeholder="พิมพ์ชื่อหนังที่ชอบ เช่น Inception...",
                             description="หนังที่ชอบ:",
                             style={"description_width":"80px"},
                             layout=widgets.Layout(width="380px"))
    rec_k     = widgets.IntSlider(value=5, min=3, max=10, step=1,
                                  description="แนะนำ:",
                                  style={"description_width":"60px"})
    rec_btn   = widgets.Button(description="🤖 แนะนำด้วย KNN",
                               button_style="success",
                               layout=widgets.Layout(width="200px"))

    def on_rec_click(_):
        with rec_out:
            clear_output()
            query = rec_input.value.strip()
            if not query:
                display(HTML("<p style='color:#e50914'>⚠️ กรุณาพิมพ์ชื่อหนัง</p>"))
                return

            movie_row, recs = get_recommendations(
                query, movies_clean, knn_model, feature_matrix, top_n=rec_k.value
            )

            if movie_row is None:
                display(HTML(f"<p style='color:#888'>ไม่พบหนัง \"{query}\" ลองพิมพ์ชื่ออื่น</p>"))
                return

            html = f"""
            <div class='section-header'>🎯 เพราะคุณชอบ...</div>
            <div class='movie-card' style='border-color:#f5c518'>
              <div class='card-left'>
                <div>
                  <div class='movie-title' style='font-size:1.1em'>{movie_row['title']}</div>
                  <div class='movie-genres'>{movie_row['genres']}</div>
                </div>
              </div>
              <span class='badge-rating'>⭐ {movie_row['avg_rating']}</span>
            </div>

            <div class='section-header'>💡 คุณอาจชอบหนังเหล่านี้ (KNN)</div>
            <div class='info-box'>⚙️ <b>KNN (K-Nearest Neighbors)</b>: สร้าง feature vector จาก genres + avg_rating
            วัด cosine distance ระหว่างหนัง แล้วเลือก {rec_k.value} เรื่องที่ใกล้ที่สุด</div>
            """

            for i, m in enumerate(recs, 1):
                html += f"""
                <div class='movie-card'>
                  <div class='card-left'>
                    <span class='rank-num'>#{i}</span>
                    <div>
                      <div class='movie-title'>{m['title']}</div>
                      <div class='movie-genres'>{m['genres']} &nbsp;·&nbsp; ⭐ {m['avg_rating']}</div>
                    </div>
                  </div>
                  <span class='badge-sim'>🎯 {m['similarity']}%</span>
                </div>"""

            display(HTML(html))

    rec_btn.on_click(on_rec_click)
    tab3 = widgets.VBox([
        widgets.HBox([rec_input, rec_k]),
        widgets.HBox([rec_btn]),
        rec_out
    ])

    # ═══════════════════════════════════════════════════════
    #  รวม Tabs
    # ═══════════════════════════════════════════════════════
    tab.children = [tab1, tab2, tab3]
    tab.set_title(0, "🏆 Top Rated (Merge Sort)")
    tab.set_title(1, "🔍 ค้นหา (Binary Search)")
    tab.set_title(2, "🤖 แนะนำ (KNN)")

    display(tab)

    # Auto-run Tab 1 เพื่อแสดงผลตั้งแต่เปิด
    on_sort_click(None)


# ── CELL 8: รันโปรแกรม ───────────────────────────────────────
create_ui()